In [36]:
listing = {
    "title": "",
    "city": "Barcelona",
    "duration_minutes": 180,
    "price": -10,
    "meeting_point": ""
}

listing_a = {
    "title": "Barcelona Tour",
    "city": "Barcelona",
    "duration_minutes": 180,
    "price": 50,
    "meeting_point": "La Rambla"
}

listing_b = {
    "title": "Barcelona Tour",
    "city": "Barcelona",
    "duration_minutes": 180,
    "price": 50,
    "meeting_point": ""
}

listing_c = {
    "title": "",
    "city": "Barcelona",
    "duration_minutes": 180,
    "price": -10,
    "meeting_point": ""
}

issue = []

In [37]:
if not listing["title"]:
    print("MISSING_TITLE")
    issue.append("MISSING_TITLE")

if not listing["meeting_point"]:
    print("MISSING_MEETING_POINT")
    issue.append("MISSING_MEETING_POINT")

if listing["price"] < 0:
    print("INVALID_PRICE")
    issue.append("INVALID_PRICE")

print(issue)

MISSING_TITLE
MISSING_MEETING_POINT
INVALID_PRICE
['MISSING_TITLE', 'MISSING_MEETING_POINT', 'INVALID_PRICE']


In [54]:
def validator(listing):
    issue = []
    
    if not listing["title"]:
        print("MISSING_TITLE")
        issue.append("MISSING_TITLE")

    if not listing["meeting_point"]:
        print("MISSING_MEETING_POINT")
        issue.append("MISSING_MEETING_POINT")

    if listing["price"] < 0:
        print("INVALID_PRICE")
        issue.append("INVALID_PRICE")

    print(issue)

In [55]:
print("listing a result")
validator(listing_a)

print("listing b result")
validator(listing_b)

print("listing c result")
validator(listing_c)

listing a result
[]
listing b result
MISSING_MEETING_POINT
['MISSING_MEETING_POINT']
listing c result
MISSING_TITLE
MISSING_MEETING_POINT
INVALID_PRICE
['MISSING_TITLE', 'MISSING_MEETING_POINT', 'INVALID_PRICE']


In [56]:
%pip install pydantic

In [31]:
# typed data modeling
raw_listing = {
    "title": "Barcelona Gothic Quarter Tour",
    "city": "Barcelona",
    "duration_minutes": "180",
    "price": "45.50",
    "meeting_point": None,
    "tags": ["walking", "history"]
}

# duration_minutes → int
# price → float
# meeting_point → Allows None
# Remaining values → Unchanged

def normalize_listing(data:dict) -> dict:
    if data["duration_minutes"] is not int:
        data["duration_minutes"] = int(data["duration_minutes"])

    if data["price"] is not float:
        data["price"] = float(data["price"])

    return data
    
    
# data.element vs data[element]

normalized = normalize_listing(raw_listing)

print(normalized)
print(type(normalized["duration_minutes"]))
print(type(normalized["price"]))


from pydantic import BaseModel

class Listing(BaseModel):
    title: str
    city: str
    duration_minutes: int
    price: float
    meeting_point: str | None = None
    tags: list[str] = []

listing = Listing(**raw_listing)
listing


# bad_listing = {
#     "title": "Barcelona Tour",
#     "city": "Barcelona",
#     "duration_minutes": "three hours",
#     "price": "free",
#     "meeting_point": None,
#     "tags": []
# }

# normalize_listing(bad_listing) 

{'title': 'Barcelona Gothic Quarter Tour', 'city': 'Barcelona', 'duration_minutes': 180, 'price': 45.5, 'meeting_point': None, 'tags': ['walking', 'history']}
<class 'int'>
<class 'float'>


Listing(title='Barcelona Gothic Quarter Tour', city='Barcelona', duration_minutes=180, price=45.5, meeting_point=None, tags=['walking', 'history'])

In [ ]:
# listing = Listing(**bad_listing)
# listing

# error

# duration_minutes
#   Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='three hours', input_type=str]
#     For further information visit https://errors.pydantic.dev/2.12/v/int_parsing
# price
#   Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='free', input_type=str]
#     For further information visit https://errors.pydantic.dev/2.12/v/float_parsing

# Why use Pydantic instead of a plain dictionary?

# 1. Problem with a plain dict
# A plain dictionary does not define or validate the expected types and constraints of its fields.

# 2. What does Pydantic BaseModel provide?
# It allows us to define a data schema and validation rules. It can also convert compatible input values into the expected types.

# 3. Why is this important for an API server?
# External input may not match the structure or types expected by the API. Pydantic validates incoming data, converts compatible values, and rejects invalid input with clear validation errors.

# Important distinction

# Valid JSON does not necessarily mean valid API input.
# JSON parsing checks whether the JSON syntax is valid.
# Pydantic validation checks whether the data matches the schema expected by our application.


In [44]:
from pydantic import BaseModel, Field, field_validator


weird_listing = {
    "title": "Barcelona Tour",
    "city": "Barcelona",
    "duration_minutes": -30,
    "price": -100,
    "meeting_point": None,
    "tags": []
}



class NewListing(BaseModel):
    title: str
    city: str
    duration_minutes: int = Field(gt=0)
    price: float = Field(ge=0)
    meeting_point: str | None = None
    tags: list[str] = Field(default_factory=list)

    @field_validator("title")
    @classmethod
    def validate_title(cls,value:str) -> str:
        value = value.strip()

        if not value:
             raise ValueError(
                "title cannot be empty"
            )

        return value

# listing = NewListing(**weird_listing)
# listing

# result : ValidationError: 2 validation errors for NewListing
# duration_minutes
#   Input should be greater than 0 [type=greater_than, input_value=-30, input_type=int]
#     For further information visit https://errors.pydantic.dev/2.12/v/greater_than
# price
#   Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-100, input_type=int]
#     For further information visit https://errors.pydantic.dev/2.12/v/greater_than_equal